# Gemini generation evaluation

This notebook evaluates the final RAG stage on top of the already-evaluated retrieval system:

`question → fund-aware retrieval → bounded evidence prompt → Gemini structured output → deterministic citation validation`

Generation is deliberately separated from retrieval evaluation so answer-quality failures can be distinguished from missing evidence or transient provider/API failures.


In [ ]:
import os
import sys
import json
import time
import getpass
import importlib
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

os.environ.setdefault(
    "HF_HOME",
    str(ROOT / ".cache" / "huggingface"),
)
os.environ.setdefault(
    "HF_HUB_CACHE",
    str(ROOT / ".cache" / "huggingface" / "hub"),
)

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from fund_rag import generator
importlib.reload(generator)

print("Project root:", ROOT)


## 1. Gemini setup

Install once if needed:

`%pip install google-genai pydantic`

The API key is read from `GEMINI_API_KEY`; if it is not set, the notebook asks for it securely without printing it.


In [ ]:
# Uncomment once if needed:
# %pip install -q google-genai pydantic

API_KEY = os.environ.get("GEMINI_API_KEY")

if not API_KEY:
    API_KEY = getpass.getpass("Gemini API key: ").strip()

if not API_KEY:
    raise ValueError("Gemini API key is required.")

MODEL = os.environ.get(
    "RAG_GENERATION_MODEL",
    "gemini-3.5-flash",
)

print("Generator model:", MODEL)


## 2. Load existing retrieval + Gemini generator


In [ ]:
start = time.perf_counter()

retriever = generator.load_retriever(ROOT)
gemini = generator.GeminiGenerator(
    api_key=API_KEY,
    model=MODEL,
)
augmenter = generator.PromptAugmenter(
    max_chunks=6,
)

assistant = generator.GeminiFundAssistant(
    retriever=retriever,
    generator=gemini,
    augmenter=augmenter,
    top_k=6,
    retrieval_mode="auto",
)

print(f"Startup: {time.perf_counter() - start:.2f}s")
print("Retrieval model:", retriever.model_name)
print("FAISS vectors:", retriever.index.ntotal)


## 3. Evaluation cases

These cover English/Bulgarian questions, cross-language evidence, exact numeric QA, broad multi-chunk QA, and abstention.


In [ ]:
CASES = [
    {
        "name": "rights_transfer_en",
        "query": "Can I transfer my fund units to another person?",
        "expected_status": "supported",
        "expected_document": "11074.pdf",
        "expected_any_terms": ["transfer", "transferred"],
    },
    {
        "name": "rights_transfer_bg",
        "query": "Мога ли да прехвърля дяловете си на друго лице?",
        "expected_status": "supported",
        "expected_document": "11074.pdf",
        "expected_any_terms": ["прехвър"],
    },
    {
        "name": "rights_general_en",
        "query": "What rights do I have as an investor?",
        "expected_status": "supported",
        "expected_document": "11074.pdf",
    },
    {
        "name": "rights_general_bg",
        "query": "Какви права имам като инвеститор?",
        "expected_status": "supported",
        "expected_document": "11074.pdf",
    },
    {
        "name": "ubb_sustainable_en",
        "query": (
            "What is the minimum percentage of sustainable investments required for "
            "ОББ ЕкспертИйз Дефанзивен Консервативен Отговорно Инвестиращ?"
        ),
        "expected_status": "supported",
        "expected_any_terms": ["20%", "20 %", "20,00"],
    },
    {
        "name": "ubb_sustainable_bg",
        "query": (
            "Какъв е минималният процент устойчиви инвестиции за "
            "ОББ ЕкспертИйз Дефанзивен Консервативен Отговорно Инвестиращ?"
        ),
        "expected_status": "supported",
        "expected_any_terms": ["20%", "20 %", "20,00"],
    },
    {
        "name": "kh_health_en",
        "query": "What is the investment objective of K&H egészség?",
        "expected_status": "supported",
    },
    {
        "name": "kh_health_bg",
        "query": "Каква е инвестиционната цел на K&H egészség?",
        "expected_status": "supported",
    },
    {
        "name": "enhanced_esg",
        "query": (
            "How does Enhanced Intelligence Global Allocation "
            "Responsible Investing evaluate ESG risk for companies?"
        ),
        "expected_status": "supported",
        "expected_any_terms": ["40", "ESG", "risk"],
    },
    {
        "name": "unsupported_weather",
        "query": "What is today's weather in Sofia?",
        "expected_status": "not_found",
    },
]


## 4. Evaluate prompt augmentation

This verifies the bridge between retrieval and generation independently of Gemini.


In [ ]:
prompt_rows = []
prompt_cache = {}

for case in CASES:
    retrieved = assistant.retrieve(case["query"])
    bundle = augmenter.build(case["query"], retrieved)

    prompt_cache[case["name"]] = {
        "retrieved": retrieved,
        "bundle": bundle,
    }

    evidence_ids = list(bundle.evidence_map)

    prompt_rows.append({
        "case": case["name"],
        "retrieved_chunks": len(retrieved),
        "prompt_chunks": len(evidence_ids),
        "query_present": case["query"] in bundle.prompt,
        "unique_evidence_ids": len(evidence_ids) == len(set(evidence_ids)),
        "has_document_metadata": all(
            item.get("document") is not None
            for item in bundle.evidence_map.values()
        ),
        "prompt_chars": len(bundle.prompt),
    })

prompt_eval = pd.DataFrame(prompt_rows)
display(prompt_eval)


# Multi-fund questions are retrieved independently per fund and interleaved before
# the six-chunk prompt budget is applied, preventing the first fund from dominating.
multi_fund_query = (
    "Compare the investment objectives of K&H egészség and K&H egészség 2."
)
multi_retrieved = assistant.retrieve(multi_fund_query)
multi_prompt = augmenter.build(multi_fund_query, multi_retrieved)
multi_prompt_funds = {
    item.get("fund")
    for item in multi_prompt.evidence_map.values()
    if item.get("fund")
}
assert {"K&H egészség", "K&H egészség 2"}.issubset(multi_prompt_funds)
print("Multi-fund prompt balance: PASS", multi_prompt_funds)


## 5. Inspect one actual Gemini prompt


In [ ]:
example = prompt_cache["rights_transfer_en"]["bundle"]

print(example.prompt)


## 6. Run Gemini generation

The Gemini API uses Pydantic structured output, so we do not depend on fragile free-form JSON parsing. Each returned evidence ID is then validated against the evidence actually supplied to the model.

Provider errors (for example a temporary HTTP 503 due to model demand) are recorded separately and **do not count as RAG answer-quality failures**.


In [ ]:
def norm(value):
    return " ".join(str(value or "").casefold().split())

records = []
rows = []

for case in CASES:
    start = time.perf_counter()

    try:
        result = assistant.answer(
            case["query"],
            include_debug=True,
        )
        latency_s = time.perf_counter() - start

        record = {
            "case": case["name"],
            "latency_s": latency_s,
            **result.to_dict(
                include_retrieved=True,
                include_debug=True,
            ),
        }
        records.append(record)

        expected_terms = case.get("expected_any_terms", [])
        answer_terms_ok = (
            True
            if not expected_terms
            else any(
                norm(term) in norm(record["answer"])
                for term in expected_terms
            )
        )

        expected_document = case.get("expected_document")
        document_ok = (
            True
            if not expected_document
            else any(
                source.get("document") == expected_document
                for source in record["sources"]
            )
        )

        retrieved_ids = {
            item.get("chunk_id")
            for item in record["retrieved"]
        }
        citation_valid = all(
            source.get("chunk_id") in retrieved_ids
            for source in record["sources"]
        )

        rows.append({
            "case": case["name"],
            "api_ok": True,
            "status": record["status"],
            "status_ok": record["status"] == case["expected_status"],
            "answer_terms_ok": answer_terms_ok,
            "expected_document_ok": document_ok,
            "citation_valid": citation_valid,
            "num_sources": len(record["sources"]),
            "latency_s": latency_s,
            "error": None,
        })

    except Exception as exc:
        latency_s = time.perf_counter() - start
        print(f"{case['name']} provider/generation failure: {exc}")
        rows.append({
            "case": case["name"],
            "api_ok": False,
            "status": "API_ERROR",
            "status_ok": False,
            "answer_terms_ok": False,
            "expected_document_ok": False,
            "citation_valid": False,
            "num_sources": 0,
            "latency_s": latency_s,
            "error": str(exc),
        })

generation_eval = pd.DataFrame(rows)
generation_eval["all_checks"] = generation_eval[
    [
        "status_ok",
        "answer_terms_ok",
        "expected_document_ok",
        "citation_valid",
    ]
].all(axis=1)

display(generation_eval)

successful = generation_eval["api_ok"]
summary = {
    "cases": len(generation_eval),
    "successful_api_calls": int(successful.sum()),
    "api_failures": int((~successful).sum()),
    "pass_rate_successful": (
        generation_eval.loc[successful, "all_checks"].mean()
        if successful.any() else None
    ),
    "status_accuracy_successful": (
        generation_eval.loc[successful, "status_ok"].mean()
        if successful.any() else None
    ),
    "citation_validity_successful": (
        generation_eval.loc[successful, "citation_valid"].mean()
        if successful.any() else None
    ),
    "mean_latency_s": generation_eval["latency_s"].mean(),
}
display(pd.DataFrame([summary]))


## 7. Inspect successful prompts, raw Gemini JSON, answers, and sources

This makes errors attributable: retrieval evidence is visible, model output is visible, and only validated evidence IDs become citations.


In [ ]:
for record in records:
    print("\n" + "=" * 110)
    print("CASE:", record["case"])
    print("QUESTION:", record["query"])

    print("\nPROMPT SENT TO GEMINI")
    print("-" * 110)
    print(record["prompt"])

    print("\nRAW GEMINI OUTPUT")
    print("-" * 110)
    print(record["raw_model_output"])

    print("\nFINAL ANSWER")
    print("-" * 110)
    print(record["answer"])

    print("\nSTATUS:", record["status"])
    print("SOURCES:")
    for source in record["sources"]:
        print(
            " -",
            source.get("document"),
            "| page", source.get("page"),
            "| section:", source.get("section"),
            "| fund:", source.get("fund"),
        )

    print(f"LATENCY: {record['latency_s']:.2f}s")


## Conclusion

Prompt augmentation is bounded (`max_chunks=6`, `max_chars_per_chunk=4500`, `max_total_evidence_chars=22000`) and the benchmark prompts observed in the last run were roughly **7k–13k characters**, well below that ceiling. The model selects supporting evidence IDs from the retrieved candidates; Python validates those IDs and supplies document/page/section metadata, so the model never invents citation locations.

A transient Gemini `503 UNAVAILABLE` was observed during evaluation and is now reported as a provider failure rather than aborting or being misclassified as a RAG-quality failure. The interactive assistant itself was manually validated on English and Bulgarian questions, including the investor-transfer and sustainable-investment examples.
